In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd

from src.data_prep import build_dataset, FEATURES, FEATURES_SMALL
from src.evaluation import evaluate, bootstrap_auc
from src.models import make_model

train = build_dataset([2014, 2015, 2016])
val = build_dataset([2017])

FEATURES_BIG = FEATURES + ["has_loss"]
print("Träning:", train.shape, " Validering:", val.shape)

Träning: (11549, 21)  Validering: (4127, 21)


In [2]:
def run(name, features, winsor=False, sector=False, label=None):
    """Tränar en modell på 2014–2016 och utvärderar den på 2017."""
    columns = features + (["Sector"] if sector else [])
    model = make_model(name, features, winsor=winsor, sector=sector)
    model.fit(train[columns], train["above_median"])
    proba = model.predict_proba(val[columns])[:, 1]

    result = evaluate(val, proba, label or name)
    low, high = bootstrap_auc(val, proba)
    result["AUC 95 %-intervall"] = f"[{low:.3f}, {high:.3f}]"
    return result

MODELS = ["Logistisk regression", "Random forest", "Gradient boosting"]

## Baseline

In [3]:
rng = np.random.default_rng(42)
random_proba = rng.random(len(val))

# P/B-regeln: lägst P/B får högst poäng. Saknat P/B hamnar i mitten.
pb_proba = val["PB ratio"].rank(ascending=False, pct=True).fillna(0.5).to_numpy()

pd.DataFrame([
    evaluate(val, random_proba, "Slump"),
    evaluate(val, pb_proba, "Lågt P/B"),
])

,Accuracy,ROC-AUC,Topp 10 % (median avk.),Botten 10 % (median avk.),Alla (median avk.)
Slump,0.499,0.488,-16.563,-13.254,-16.319
Lågt P/B,0.476,0.484,-19.926,-11.082,-16.319


## Experiment A: Vilken modell? (15 nyckeltal, ingen winsorisering, ingen sektor)

In [4]:
pd.DataFrame([run(m, FEATURES_BIG) for m in MODELS])

,Accuracy,ROC-AUC,Topp 10 % (median avk.),Botten 10 % (median avk.),Alla (median avk.),AUC 95 %-intervall
Logistisk regression,0.572,0.610,-9.901,-43.031,-16.319,"[0.592, 0.627]"
Random forest,0.585,0.618,-10.077,-45.116,-16.319,"[0.601, 0.636]"
Gradient boosting,0.583,0.616,-10.775,-45.116,-16.319,"[0.599, 0.634]"


## Experiment B: Hjälper winsorisering? (Henris och Marcies idé)

In [5]:
pd.DataFrame([
    run(m, FEATURES_BIG, winsor=w, label=f"{m} | {'med' if w else 'utan'} winsor")
    for m in MODELS for w in [False, True]
])

,Accuracy,ROC-AUC,Topp 10 % (median avk.),Botten 10 % (median avk.),Alla (median avk.),AUC 95 %-intervall
Logistisk regression | utan winsor,0.572,0.610,-9.901,-43.031,-16.319,"[0.592, 0.627]"
Logistisk regression | med winsor,0.583,0.621,-13.124,-43.339,-16.319,"[0.604, 0.638]"
Random forest | utan winsor,0.585,0.618,-10.077,-45.116,-16.319,"[0.601, 0.636]"
Random forest | med winsor,0.579,0.622,-10.020,-45.302,-16.319,"[0.605, 0.641]"
Gradient boosting | utan winsor,0.583,0.616,-10.775,-45.116,-16.319,"[0.599, 0.634]"
Gradient boosting | med winsor,0.580,0.617,-10.775,-45.302,-16.319,"[0.600, 0.635]"


## Experiment C: 5 eller 15 nyckeltal? (med winsorisering)

In [6]:
pd.DataFrame([
    run(m, f, winsor=True, label=f"{m} | {n} nyckeltal")
    for m in MODELS for n, f in [(5, FEATURES_SMALL), (15, FEATURES_BIG)]
])

,Accuracy,ROC-AUC,Topp 10 % (median avk.),Botten 10 % (median avk.),Alla (median avk.),AUC 95 %-intervall
Logistisk regression | 5 nyckeltal,0.575,0.606,-12.712,-42.562,-16.319,"[0.589, 0.623]"
Logistisk regression | 15 nyckeltal,0.583,0.621,-13.124,-43.339,-16.319,"[0.604, 0.638]"
Random forest | 5 nyckeltal,0.564,0.600,-10.649,-47.525,-16.319,"[0.584, 0.619]"
Random forest | 15 nyckeltal,0.579,0.622,-10.020,-45.302,-16.319,"[0.605, 0.641]"
Gradient boosting | 5 nyckeltal,0.562,0.596,-11.739,-44.264,-16.319,"[0.578, 0.614]"
Gradient boosting | 15 nyckeltal,0.580,0.617,-10.775,-45.302,-16.319,"[0.600, 0.635]"


## Experiment D: Hjälper sektor? (15 nyckeltal, med winsorisering)

In [7]:
pd.DataFrame([
    run(m, FEATURES_BIG, winsor=True, sector=s, label=f"{m} | {'med' if s else 'utan'} sektor")
    for m in MODELS for s in [False, True]
])

,Accuracy,ROC-AUC,Topp 10 % (median avk.),Botten 10 % (median avk.),Alla (median avk.),AUC 95 %-intervall
Logistisk regression | utan sektor,0.583,0.621,-13.124,-43.339,-16.319,"[0.604, 0.638]"
Logistisk regression | med sektor,0.596,0.626,-13.456,-39.285,-16.319,"[0.610, 0.643]"
Random forest | utan sektor,0.579,0.622,-10.020,-45.302,-16.319,"[0.605, 0.641]"
Random forest | med sektor,0.598,0.632,-9.196,-45.116,-16.319,"[0.615, 0.649]"
Gradient boosting | utan sektor,0.580,0.617,-10.775,-45.302,-16.319,"[0.600, 0.635]"
Gradient boosting | med sektor,0.596,0.633,-8.879,-40.250,-16.319,"[0.616, 0.650]"
